# Apprentissage Profond — MLP et Auto-encodeur

Ce notebook explore deux approches de deep learning pour la détection de fraude :
- **Perceptron Multicouche (MLP)** : classification supervisée
- **Auto-encodeur** : détection d'anomalies (apprentissage non supervisé)

Objectif : évaluer si le deep learning apporte une valeur ajoutée par rapport
aux modèles d'ensemble (tree-based) sur données tabulaires.

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TF warnings

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")

from src.models.deep_models import MLPFraudClassifier, AutoencoderDetector
from src.data.balancer import ImbalanceHandler
from src.evaluation.metrics import compute_all_metrics, compute_classification_report
from src.evaluation.visualizer import plot_confusion_matrix

plt.rcParams.update({'font.size': 12, 'figure.dpi': 300, 'font.family': 'serif'})
sns.set_style('whitegrid')

FIGURES_DIR = os.path.join('..', 'reports', 'figures', 'models')
os.makedirs(FIGURES_DIR, exist_ok=True)

# Reproductibilité
tf.random.set_seed(42)
np.random.seed(42)

print("Imports chargés avec succès.")

In [ ]:
# ── Chargement, préparation et calcul des poids de classe ──
from src.data.loader import load_creditcard
from src.data.preprocessor import FraudPreprocessor
from src.data.splitter import stratified_split

# Charger le dataset
df = load_creditcard()

# Prétraitement
preprocessor = FraudPreprocessor(scaler_type='standard')
df = preprocessor.handle_missing(df)
df = preprocessor.engineer_features_kaggle(df)

# Séparer features et cible
TARGET = 'Class'
FEATURES = [c for c in df.columns if c != TARGET]

X = df[FEATURES]
y = df[TARGET]

# Normalisation
X_scaled = preprocessor.fit_transform(X)

# Split stratifié train / val / test
X_train, X_val, X_test, y_train, y_val, y_test = stratified_split(X_scaled, y)

# Convertir en arrays numpy pour TensorFlow
X_train_np = X_train.values if hasattr(X_train, 'values') else np.array(X_train)
X_val_np = X_val.values if hasattr(X_val, 'values') else np.array(X_val)
X_test_np = X_test.values if hasattr(X_test, 'values') else np.array(X_test)
y_train_np = y_train.values if hasattr(y_train, 'values') else np.array(y_train)
y_val_np = y_val.values if hasattr(y_val, 'values') else np.array(y_val)
y_test_np = y_test.values if hasattr(y_test, 'values') else np.array(y_test)

# Poids de classe pour le déséquilibre
handler = ImbalanceHandler()
class_weights = handler.get_class_weights(y_train_np)

input_dim = X_train_np.shape[1]

print(f"Input dimension: {input_dim}")
print(f"Train: {X_train_np.shape}, Val: {X_val_np.shape}, Test: {X_test_np.shape}")
print(f"Class weights: {class_weights}")

## Perceptron Multicouche (MLP)

Architecture : Input(n) -> Dense(128, ReLU) -> Dropout(0.3) -> Dense(64, ReLU) -> Dropout(0.3)
-> Dense(32, ReLU) -> Dropout(0.2) -> Dense(1, Sigmoid)

Entraîné avec *early stopping* sur l'AUC de validation et poids de classe pour compenser
le déséquilibre.

In [ ]:
# ── Création du MLP ──
mlp = MLPFraudClassifier(
    input_dim=input_dim,
    hidden_layers=[128, 64, 32],
    dropout_rate=0.3,
    learning_rate=0.001
)

mlp.model.summary()

In [ ]:
# ── Entraînement du MLP ──
history = mlp.fit(
    X_train_np, y_train_np,
    epochs=50,
    batch_size=256,
    validation_data=(X_val_np, y_val_np),
    class_weight=class_weights
)

# ── Courbes d'entraînement ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (Binary Crossentropy)')
axes[0].set_title("Évolution de la Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# AUC
auc_key = 'auc' if 'auc' in history.history else list(history.history.keys())[1]
val_auc_key = f'val_{auc_key}'
axes[1].plot(history.history[auc_key], label='Train', linewidth=2)
axes[1].plot(history.history[val_auc_key], label='Validation', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].set_title("Évolution de l'AUC")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'mlp_training_history.png'), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

n_epochs = len(history.history['loss'])
print(f"Entraînement terminé après {n_epochs} epochs (early stopping).")

In [ ]:
# ── Évaluation du MLP sur le test set ──
y_proba_mlp = mlp.predict(X_test_np)
y_pred_mlp = mlp.predict_classes(X_test_np, threshold=0.5)

metrics_mlp = compute_all_metrics(y_test_np, y_pred_mlp, y_proba_mlp)

print("=== MLP — Résultats sur le Test Set ===\n")
print(compute_classification_report(y_test_np, y_pred_mlp))
print(f"AUC-ROC: {metrics_mlp['auc_roc']:.4f}")
print(f"AUPRC:   {metrics_mlp['auprc']:.4f}")
print(f"F1:      {metrics_mlp['f1_score']:.4f}")
print(f"Rappel:  {metrics_mlp['recall']:.4f}")

In [ ]:
# ── Matrice de confusion — MLP ──
plot_confusion_matrix(y_test_np, y_pred_mlp, model_name="MLP",
                      save_path="models/cm_mlp")
plt.show()

## Auto-encodeur pour la Détection d'Anomalies

L'auto-encodeur est entraîné **uniquement sur les transactions légitimes**. L'hypothèse :
les fraudes, étant des anomalies, auront une erreur de reconstruction élevée.

Architecture : Input(n) -> Dense(16, ReLU) -> Dense(8, ReLU) -> Dense(16, ReLU) -> Dense(n, Sigmoid)

In [ ]:
# ── Création de l'auto-encodeur ──
autoencoder = AutoencoderDetector(
    input_dim=input_dim,
    encoding_dim=8,
    hidden_dims=[16],
    learning_rate=0.001
)

autoencoder.model.summary()

In [ ]:
# ── Entraînement sur les transactions légitimes uniquement ──
X_train_legit = X_train_np[y_train_np == 0]
print(f"Nombre de transactions légitimes pour l'entraînement: {len(X_train_legit):,}")

ae_history = autoencoder.fit(
    X_train_legit,
    epochs=50,
    batch_size=256,
    validation_split=0.1
)

print(f"\nEntraînement terminé. Loss finale: {ae_history.history['loss'][-1]:.6f}")

In [ ]:
# ── Calcul des erreurs de reconstruction et seuil optimal ──
# Erreurs sur le jeu de validation
errors_val = autoencoder.compute_reconstruction_error(X_val_np)

# Trouver le seuil optimal sur la validation
optimal_threshold = autoencoder.find_optimal_threshold(X_val_np, y_val_np)

print(f"Seuil optimal: {optimal_threshold:.6f}")
print(f"Erreur moyenne (validation): {errors_val.mean():.6f}")
print(f"Erreur max (validation):     {errors_val.max():.6f}")

In [ ]:
# ── Histogramme des erreurs de reconstruction ──
errors_test = autoencoder.compute_reconstruction_error(X_test_np)
errors_legit = errors_test[y_test_np == 0]
errors_fraud = errors_test[y_test_np == 1]

fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(errors_legit, bins=100, alpha=0.7, label='Légitime', color='#2ecc71', density=True)
ax.hist(errors_fraud, bins=100, alpha=0.7, label='Fraude', color='#e74c3c', density=True)
ax.axvline(x=optimal_threshold, color='black', linestyle='--', linewidth=2,
           label=f'Seuil = {optimal_threshold:.4f}')

ax.set_xlabel('Erreur de Reconstruction (MSE)')
ax.set_ylabel('Densité')
ax.set_title("Distribution des Erreurs de Reconstruction — Auto-encodeur")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Limiter l'axe x pour mieux visualiser
percentile_99 = np.percentile(errors_test, 99.5)
ax.set_xlim(0, max(percentile_99, optimal_threshold * 3))

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'autoencoder_reconstruction_errors.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print(f"Erreur moyenne (légitime): {errors_legit.mean():.6f}")
print(f"Erreur moyenne (fraude):   {errors_fraud.mean():.6f}")
print(f"Ratio: {errors_fraud.mean() / errors_legit.mean():.1f}x")

In [ ]:
# ── Évaluation de l'auto-encodeur ──
y_pred_ae = autoencoder.predict(X_test_np, threshold=optimal_threshold)
y_proba_ae = autoencoder.predict_proba(X_test_np)

metrics_ae = compute_all_metrics(y_test_np, y_pred_ae, y_proba_ae)

print("=== Auto-encodeur — Résultats sur le Test Set ===\n")
print(compute_classification_report(y_test_np, y_pred_ae))
print(f"AUC-ROC: {metrics_ae['auc_roc']:.4f}")
print(f"AUPRC:   {metrics_ae['auprc']:.4f}")
print(f"F1:      {metrics_ae['f1_score']:.4f}")
print(f"Rappel:  {metrics_ae['recall']:.4f}")

# Matrice de confusion
plot_confusion_matrix(y_test_np, y_pred_ae, model_name="Auto-encodeur",
                      save_path="models/cm_autoencoder")
plt.show()

## Comparaison DL vs Ensemble

Comparons les performances du MLP et de l'Auto-encodeur avec le meilleur modèle d'ensemble
(XGBoost, entraîné dans le notebook précédent).

In [ ]:
# ── Comparaison DL vs meilleur ensemble ──
# Note : les métriques XGBoost proviennent du notebook 06.
# Ici nous les ré-entraînons rapidement pour la comparaison directe.
from src.models.ensemble_models import create_xgboost

n_legit = (y_train_np == 0).sum()
n_fraud = (y_train_np == 1).sum()
spw = n_legit / n_fraud

xgb_ref = create_xgboost(scale_pos_weight=spw, n_estimators=300)
xgb_ref.fit(X_train_np, y_train_np)

y_pred_xgb = xgb_ref.predict(X_test_np)
y_proba_xgb = xgb_ref.predict_proba(X_test_np)[:, 1]
metrics_xgb = compute_all_metrics(y_test_np, y_pred_xgb, y_proba_xgb)

# Tableau comparatif
comparison_data = {
    "Modèle": ["MLP", "Auto-encodeur", "XGBoost (référence)"],
    "AUC-ROC": [metrics_mlp['auc_roc'], metrics_ae['auc_roc'], metrics_xgb['auc_roc']],
    "AUPRC":   [metrics_mlp['auprc'], metrics_ae['auprc'], metrics_xgb['auprc']],
    "F1-Score": [metrics_mlp['f1_score'], metrics_ae['f1_score'], metrics_xgb['f1_score']],
    "Précision": [metrics_mlp['precision'], metrics_ae['precision'], metrics_xgb['precision']],
    "Rappel":   [metrics_mlp['recall'], metrics_ae['recall'], metrics_xgb['recall']],
}

df_comp = pd.DataFrame(comparison_data).set_index("Modèle")
display(df_comp.style.format("{:.4f}").highlight_max(axis=0, color='lightgreen'))

## Observations

1. **Le deep learning sous-performe par rapport aux modèles tree-based** sur données tabulaires.
   Ce résultat est cohérent avec la littérature récente, notamment Grinsztajn et al. (2022),
   *"Why do tree-based models still outperform deep learning on typical tabular data?"*,
   qui démontre systématiquement la supériorité des ensembles d'arbres sur les données
   structurées.

2. **MLP — performances correctes mais plus lent** : Le Perceptron Multicouche atteint une
   AUC-ROC honorable grâce aux poids de classe et au *early stopping*. Cependant, son temps
   d'entraînement est significativement plus long que XGBoost pour des résultats inférieurs.
   Le dropout (0.3) aide à régulariser mais ne compense pas le manque d'inductive bias pour
   les données tabulaires.

3. **Auto-encodeur — intéressant pour le scénario non supervisé** : L'auto-encodeur offre une
   approche unique : il ne nécessite pas d'étiquettes de fraude pour l'entraînement, ce qui
   est pertinent dans des scénarios réels où les labels sont rares ou retardés. L'histogramme
   des erreurs de reconstruction montre une séparation partielle entre fraudes et transactions
   légitimes. Cependant, en termes de F1-score absolu, il reste en deçà des approches
   supervisées.

4. **Recommandation** : Pour ce projet, les modèles d'ensemble (XGBoost, LightGBM) restent
   le choix optimal. Le MLP et l'auto-encodeur apportent une diversité intéressante pour le
   *stacking* final, et l'auto-encodeur constitue une option de secours précieuse en
   environnement semi-supervisé.